In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

# ================================================================
# 1. Paths
# ================================================================
adloc_path = "/home/fc/gkx/DBMANet/texnet定位/local/texnet/adloc_27/adloc_events.csv"
config_path = "/home/fc/gkx/DBMANet/texnet定位/local/texnet/config.json"
original_path = "/home/fc/gkx/DBMANet/texnet定位/local/texnet/events_filtered.csv"
output_dir = "/home/fc/gkx/DBMANet/texnet定位/local/texnet/adloc_27/figures"
os.makedirs(output_dir, exist_ok=True)

# ================================================================
# 2. Load data
# ================================================================
adloc = pd.read_csv(adloc_path)
adloc['time'] = pd.to_datetime(adloc['time'])

with open(config_path, 'r') as f:
    config = json.load(f)

center_lon = config['longitude0']
center_lat = config['latitude0']
radius_deg = config['maxradius_degree']

print(f"ADLoc 27 events: {len(adloc)}")
print(f"Region center: ({center_lat:.4f}, {center_lon:.4f})")
print(f"Region radius: {radius_deg} deg")

# ================================================================
# 3. Load original catalog and filter by time (Jan 1–30)
# ================================================================
original = pd.read_csv(original_path)
original['time'] = pd.to_datetime(original['time'])
original['latitude'] = original['Latitude (WGS84)']
original['longitude'] = original['Longitude (WGS84)']
original['depth_km'] = original['Depth of Hypocenter (Km.  Rel to MSL)']

start_date = '2026-01-01'
end_date = '2026-01-30 23:59:59'
original_filtered = original[(original['time'] >= start_date) & (original['time'] <= end_date)].copy()
adloc_filtered = adloc[(adloc['time'] >= start_date) & (adloc['time'] <= end_date)].copy()

print(f"Original catalog (Jan 1-30): {len(original_filtered)}")
print(f"ADLoc 27 (Jan 1-30): {len(adloc_filtered)}")

# ================================================================
# 4. Matching function
# ================================================================
def match_events(my_cat, std_cat, time_thresh=5.0, dist_thresh=10.0):
    ref_time = min(my_cat['time'].min(), std_cat['time'].min())
    my_time = (my_cat['time'] - ref_time).dt.total_seconds().values
    std_time = (std_cat['time'] - ref_time).dt.total_seconds().values
    my_lat = my_cat['latitude'].values
    my_lon = my_cat['longitude'].values
    std_lat = std_cat['latitude'].values
    std_lon = std_cat['longitude'].values
    avg_lat = np.mean(np.concatenate([my_lat, std_lat]))
    deg_to_km_lat = 111.0
    deg_to_km_lon = 111.0 * np.cos(np.radians(avg_lat))
    matched_pairs = []
    used_my = set()
    for i in range(len(std_cat)):
        dt = np.abs(my_time - std_time[i])
        dx = (my_lon - std_lon[i]) * deg_to_km_lon
        dy = (my_lat - std_lat[i]) * deg_to_km_lat
        dist = np.sqrt(dx**2 + dy**2)
        candidates = np.where((dt < time_thresh) & (dist < dist_thresh))[0]
        candidates = [c for c in candidates if c not in used_my]
        if len(candidates) > 0:
            best_idx = candidates[np.argmin(dist[candidates])]
            used_my.add(best_idx)
            matched_pairs.append({'my_idx': best_idx, 'std_idx': i, 'dt': dt[best_idx], 'dist': dist[best_idx]})
    matched = []
    for pair in matched_pairs:
        my_row = my_cat.iloc[pair['my_idx']]
        std_row = std_cat.iloc[pair['std_idx']]
        matched.append({
            'my_time': my_row['time'], 'std_time': std_row['time'],
            'my_lat': my_row['latitude'], 'std_lat': std_row['latitude'],
            'my_lon': my_row['longitude'], 'std_lon': std_row['longitude'],
            'my_depth': my_row['depth_km'], 'std_depth': std_row['depth_km'],
            'dt': pair['dt'], 'dist': pair['dist']
        })
    return pd.DataFrame(matched)

# Try matching with default thresholds, then relaxed if needed
matched = match_events(adloc_filtered, original_filtered, time_thresh=5.0, dist_thresh=10.0)
if len(matched) == 0:
    matched = match_events(adloc_filtered, original_filtered, time_thresh=10.0, dist_thresh=15.0)
print(f"Matched events: {len(matched)}")

# ================================================================
# 5. Compute error statistics
# ================================================================
lat_err = (matched['my_lat'] - matched['std_lat']) * 111.0
lon_err = (matched['my_lon'] - matched['std_lon']) * 111.0 * np.cos(np.radians(np.mean(matched['std_lat'])))
depth_err = matched['my_depth'] - matched['std_depth']
dt_err = matched['dt']

stats_text = (
    f"Matched: {len(matched)}\n"
    f"Lat:  mean={np.mean(lat_err):.3f} km  median={np.median(lat_err):.3f} km  std={np.std(lat_err):.3f} km\n"
    f"Lon:  mean={np.mean(lon_err):.3f} km  median={np.median(lon_err):.3f} km  std={np.std(lon_err):.3f} km\n"
    f"Depth:mean={np.mean(depth_err):.3f} km  median={np.median(depth_err):.3f} km  std={np.std(depth_err):.3f} km\n"
    f"Time: mean={np.mean(dt_err):.3f} s   median={np.median(dt_err):.3f} s   std={np.std(dt_err):.3f} s"
)

# ================================================================
# 6. Plot comparison figure
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.subplots_adjust(bottom=0.32)

# Left: Epicenter comparison
ax = axes[0]
ax.scatter(matched['std_lon'], matched['std_lat'], label='Original (in circle)', s=30, alpha=0.7, c='blue')
ax.scatter(matched['my_lon'], matched['my_lat'], label='ADLoc 27', s=30, marker='x', alpha=0.7, c='red')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()
ax.set_title(f'Epicenter Comparison (matched {len(matched)} events)')
ax.grid(True, linestyle='--', alpha=0.5)

# Right: Depth comparison
ax = axes[1]
ax.scatter(matched['std_depth'], matched['my_depth'], alpha=0.7, c='green', s=30)
max_depth = max(matched['std_depth'].max(), matched['my_depth'].max()) + 1
ax.plot([0, max_depth], [0, max_depth], 'r--', label='1:1 line')
ax.set_xlabel('Original Depth (km)')
ax.set_ylabel('ADLoc Depth (km)')
ax.set_title('Depth Comparison')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_aspect('equal')
ax.set_xlim(0, max_depth)
ax.set_ylim(0, max_depth)

# Place statistics text below the subplots
fig.text(0.5, 0.015, stats_text, ha='center', fontsize=10, family='monospace',
         bbox=dict(boxstyle='round', facecolor='whitesmoke', alpha=0.9, edgecolor='gray'))

plt.tight_layout(rect=[0, 0.18, 1, 1])
plt.savefig(os.path.join(output_dir, "adloc_27_comparison.png"), dpi=300, bbox_inches='tight')
print(" adloc_27_comparison.png saved.")
plt.close()

print(f"\n Done. Figure saved to: {output_dir}/adloc_27_comparison.png")